In [1]:
import pandas as pd 
import numpy as np 

# 1. Setup and Data Loading

In [7]:
df = pd.read_excel('zepto_v1.xlsx')

# 2. Data Exploration

In [ ]:
print(f"Total Rows: {len(df)}")
display(df.head(10))

Total Rows: 3732


,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity
0,Fruits & Vegetables,Onion,2500,16,3,2100,1000,False,1
1,Fruits & Vegetables,Tomato Hybrid,4200,16,3,3500,1000,False,1
2,Fruits & Vegetables,Tender Coconut,5100,15,3,4300,58,False,1
3,Fruits & Vegetables,Coriander Leaves,2000,15,3,1700,100,False,100
4,Fruits & Vegetables,Ladies Finger,1400,14,3,1200,250,False,250
5,Fruits & Vegetables,Potato,3500,17,3,2900,1000,False,1
6,Fruits & Vegetables,Lemon,7500,16,3,6300,200,False,200
7,Fruits & Vegetables,Watermelon,5800,15,3,4900,58,False,1
8,Fruits & Vegetables,Capsicum Green,2300,17,3,1900,250,False,250
9,Fruits & Vegetables,Chilli Green,1900,15,3,1600,100,False,100


In [ ]:
# Check for Null values
print("\n--- Null Values ---")
print(df.isnull().sum())


--- Null Values ---
Category                  0
name                      0
mrp                       0
discountPercent           0
availableQuantity         0
discountedSellingPrice    0
weightInGms               0
outOfStock                0
quantity                  0
dtype: int64


In [ ]:
# Different product categories
print("\n--- Categories ---")
print(sorted(df['Category'].dropna().unique()))


--- Categories ---
['Beverages', 'Biscuits', 'Chocolates & Candies', 'Cooking Essentials', 'Dairy, Bread & Batter', 'Fruits & Vegetables', 'Health & Hygiene', 'Home & Cleaning', 'Ice Cream & Desserts', 'Meats, Fish & Eggs', 'Munchies', 'Paan Corner', 'Packaged Food', 'Personal Care']


In [ ]:
# Products in stock vs out of stock
print("\n--- Stock Status ---")
print(df['outOfStock'].value_counts())


--- Stock Status ---
outOfStock
False    3279
True      453
Name: count, dtype: int64


In [14]:
# Product names present multiple times (Updated to not use sku_id)
print("\n--- Duplicate SKUs ---")
duplicate_names = df.groupby('name').size().reset_index(name='Count')
duplicate_names = duplicate_names[duplicate_names['Count'] > 1].sort_values(by='Count', ascending=False)
display(duplicate_names.head(10))


--- Duplicate SKUs ---


,name,Count
70,Amul Delicious Fat Spread - Cholesterol Free,10
966,Mother's Recipe Tamarind Paste,10
1301,Quaker Oats,10
1338,Saffola Veggie Twist Masala Oats,10
99,Arden Eggs White,10
1450,Sunfeast Yippee! Pasta Treat - Sour Cream Onion,10
778,Kellogg's Real Almond & Honey Corn Flakes,9
71,Amul Fresh Cream,8
1677,iD Idli & Dosa Batter,7
718,Himalaya Anti Dandruff Shampoo,6


# 3. Data Cleaning

In [15]:
# Delete rows where mrp = 0
df = df[df['mrp'] != 0].copy()

In [16]:
# Convert paise to rupees
df['mrp'] = df['mrp'] / 100.0
df['discountedSellingPrice'] = df['discountedSellingPrice'] / 100.0

# 4. Data Analysis

In [27]:
# Q1. Top 10 best-value products (highest discount)
top_10_discounts = df[['name', 'mrp', 'discountPercent']].drop_duplicates().sort_values(by='discountPercent', ascending=False).head(10)
print("\n--- Q1: Top 10 Best-Value Products ---")
display(top_10_discounts)


--- Q1: Top 10 Best-Value Products ---


,name,mrp,discountPercent
2608,Dukes Waffy Chocolate Wafers,45.0,51
2619,Dukes Waffy Strawberry Wafers,45.0,51
2615,Dukes Waffy Orange Wafers,45.0,51
490,Ceres Foods Nalli Nihari Instant Liquid Masala,220.0,50
278,Ceres Foods Fish Mustard Instant Liquid Masala,220.0,50
1213,RRO Mascarpone Cheese,355.0,50
198,Chef's Basket Durum Wheat Elbow Pasta,160.0,50
2613,Dukes Waffy Strawberry Roll,150.0,50
3636,Epigamia Fruit Yogurt Vanilla,40.0,50
1609,Moi Soi Kung Pao Sauce - For Stir Fry Marinade...,280.0,50


In [28]:
# Q2. Products with High MRP (> ₹300) but Out of Stock
high_mrp_oos = df[(df['outOfStock'] == True) & (df['mrp'] > 300)][['name', 'mrp']].drop_duplicates().sort_values(by='mrp', ascending=False)
print("\n--- Q2: High MRP (> ₹300) & Out of Stock ---")
display(high_mrp_oos)


--- Q2: High MRP (> ₹300) & Out of Stock ---


,name,mrp
580,Patanjali Cow's Ghee,565.0
3096,"MamyPoko Pants Standard Diapers, Extra Large (...",399.0
553,Aashirvaad Atta With Mutigrains,315.0
573,Everest Kashmiri Lal Chilli Powder,310.0


In [29]:
# Q3. Estimated Revenue per category
df['revenue'] = df['discountedSellingPrice'] * df['availableQuantity']
category_revenue = df.groupby('Category')['revenue'].sum().sort_values(ascending=False).reset_index()
print("\n--- Q3: Estimated Revenue per Category ---")
display(category_revenue)


--- Q3: Estimated Revenue per Category ---


,Category,revenue
0,Cooking Essentials,337369.0
1,Munchies,337369.0
2,Paan Corner,270849.0
3,Personal Care,270849.0
4,Ice Cream & Desserts,224385.0
5,Chocolates & Candies,224385.0
6,Packaged Food,224385.0
7,Home & Cleaning,122661.0
8,Health & Hygiene,64180.0
9,"Dairy, Bread & Batter",55051.0


In [30]:
# Q4. MRP > ₹500 and Discount < 10%
premium_low_discount = df[(df['mrp'] > 500) & (df['discountPercent'] < 10)][['name', 'mrp', 'discountPercent']].drop_duplicates().sort_values(by=['mrp', 'discountPercent'], ascending=[False, False])
print("\n--- Q4: Premium Products (MRP > ₹500) with Low Discount (< 10%) ---")
display(premium_low_discount)


--- Q4: Premium Products (MRP > ₹500) with Low Discount (< 10%) ---


,name,mrp,discountPercent
121,Dhara Kachi Ghani Mustard Oil Jar,1250.0,8
144,Saffola Gold (Jar),1240.0,0
146,Fortune Rice Bran Health Oil (Jar),1050.0,1
224,Dhara Filtered Groundnut Oil (Jar),1050.0,1
511,Dhara Filtered Groundnut Oil (Jar),1050.0,0
114,Fortune Soyabean Oil,1005.0,0
104,Fortune Sunlite Refined Sunflower (Jar),925.0,0
3545,Surf Excel Matic Powder Front Load,810.0,7
3556,Surf Excel Matic Top Load,720.0,9
1669,Pedigree Puppy Dry Dog Food Food Chicken & Milk,690.0,6


In [31]:
# Q5. Top 5 categories with highest average discount
top_avg_discount = df.groupby('Category')['discountPercent'].mean().round(2).sort_values(ascending=False).head(5).reset_index()
print("\n--- Q5: Top 5 Categories by Average Discount ---")
display(top_avg_discount)


--- Q5: Top 5 Categories by Average Discount ---


,Category,discountPercent
0,Fruits & Vegetables,15.46
1,"Meats, Fish & Eggs",11.03
2,Ice Cream & Desserts,8.32
3,Packaged Food,8.32
4,Chocolates & Candies,8.32


In [32]:
# Q6. Price per gram for products >= 100g
df_large = df[df['weightInGms'] >= 100].copy()
df_large['price_per_gram'] = (df_large['discountedSellingPrice'] / df_large['weightInGms']).round(2)
best_value_per_gram = df_large[['name', 'weightInGms', 'discountedSellingPrice', 'price_per_gram']].drop_duplicates().sort_values(by='price_per_gram')
print("\n--- Q6: Price per gram (Best Value) for items >= 100g ---")
display(best_value_per_gram.head(10))


--- Q6: Price per gram (Best Value) for items >= 100g ---


,name,weightInGms,discountedSellingPrice,price_per_gram
0,Onion,1000,21.0,0.02
3645,Vicks Cough Drops Menthol,1160,20.0,0.02
95,Tata Salt,1000,24.0,0.02
550,Aashirvaad Iodised Salt,1000,19.0,0.02
3572,Shubh kart - Nirmal sugandhi mogra wet dhoop z...,1160,28.0,0.02
20,Onion,3000,57.0,0.02
53,Raw Banana,500,17.0,0.03
85,Carrot,500,15.0,0.03
21,Potato,3000,84.0,0.03
3554,Shubh kart - Tejas Twisted Cotton Wicks 1000n,1000,28.0,0.03


In [33]:
# Q7. Categorize by weight
conditions = [
    (df['weightInGms'] < 1000),
    (df['weightInGms'] < 5000)
]
choices = ['Low', 'Medium']
df['weight_category'] = np.select(conditions, choices, default='Bulk')
print("\n--- Q7: Weight Categories (Sample Output) ---")
display(df[['name', 'weightInGms', 'weight_category']].head(10))


--- Q7: Weight Categories (Sample Output) ---


,name,weightInGms,weight_category
0,Onion,1000,Medium
1,Tomato Hybrid,1000,Medium
2,Tender Coconut,58,Low
3,Coriander Leaves,100,Low
4,Ladies Finger,250,Low
5,Potato,1000,Medium
6,Lemon,200,Low
7,Watermelon,58,Low
8,Capsicum Green,250,Low
9,Chilli Green,100,Low


In [34]:
# Q8. Total Inventory Weight Per Category
inventory_weight = df.groupby('Category').apply(lambda x: (x['weightInGms'] * x['availableQuantity']).sum()).sort_values(ascending=False).reset_index(name='Total Weight (g)')
print("\n--- Q8: Total Inventory Weight per Category ---")
display(inventory_weight)


--- Q8: Total Inventory Weight per Category ---


C:\Users\adwic\AppData\Local\Temp\ipykernel_15532\2556159293.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  inventory_weight = df.groupby('Category').apply(lambda x: (x['weightInGms'] * x['availableQuantity']).sum()).sort_values(ascending=False).reset_index(name='Total Weight (g)')


,Category,Total Weight (g)
0,Cooking Essentials,1404654
1,Munchies,1404654
2,Ice Cream & Desserts,490797
3,Packaged Food,490797
4,Chocolates & Candies,490797
5,Home & Cleaning,373161
6,Paan Corner,348187
7,Personal Care,348187
8,Beverages,143735
9,"Dairy, Bread & Batter",143735
